# Transformation des fichiers Excel historiques vers les tables Polypbase

Notebook corrigé : les anomalies détectées sont exportées dans `exports_polypbase/anomalies/` et exclues automatiquement des CSV finaux.

In [233]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Dossier du projet.
# Ce chemin doit pointer vers le clone propre du projet, pas vers la corbeille.
PROJECT_DIR = Path("/Users/akkouh/Desktop/POLYPBASE")

# On cherche les fichiers Excel dans plusieurs emplacements possibles.
# Le premier dossier contenant des fichiers Suivi_*.xlsx sera utilisé.
data_dir_candidates = [
    PROJECT_DIR / "data",
    PROJECT_DIR,
    Path("/Users/akkouh/Desktop/polybase"),
]

DATA_DIR = None
excel_files = []

for candidate in data_dir_candidates:
    files = sorted(candidate.glob("Suivi_*.xlsx")) if candidate.exists() else []
    if files:
        DATA_DIR = candidate
        excel_files = files
        break

# Si aucun fichier Excel n'est trouvé, on garde data/ par défaut.
if DATA_DIR is None:
    DATA_DIR = PROJECT_DIR / "data"
    excel_files = []

# Dossiers de sortie pour les tables propres et les anomalies.
OUTPUT_DIR = PROJECT_DIR / "exports_polypbase"
ANOMALY_DIR = OUTPUT_DIR / "anomalies"

OUTPUT_DIR.mkdir(exist_ok=True)
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier projet :", PROJECT_DIR)
print("Dossier données :", DATA_DIR)
print("Nombre de fichiers Excel trouvés :", len(excel_files))
display([file.name for file in excel_files])


Dossier projet : /Users/akkouh/Desktop/POLYPBASE
Dossier données : /Users/akkouh/Desktop/polybase
Nombre de fichiers Excel trouvés : 8


['Suivi_2019.xlsx',
 'Suivi_2020.xlsx',
 'Suivi_2021.xlsx',
 'Suivi_2022.xlsx',
 'Suivi_2023.xlsx',
 'Suivi_2024.xlsx',
 'Suivi_2025.xlsx',
 'Suivi_2026.xlsx']

In [234]:
# Vérification rapide des feuilles disponibles dans chaque fichier.

for file in excel_files:
    xls = pd.ExcelFile(file)
    print(f"\n{file.name}")
    print(xls.sheet_names)



Suivi_2019.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2020.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2021.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2022.xlsx
['Aurelia', 'Cubozoa', 'Hydrozoa', 'Rhizostomae', 'Semaestomeae', 'Coronatae']

Suivi_2023.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomeae', 'Semaestomeae', 'Coronatae', 'UMR_MARBEC']

Suivi_2024.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Semaestomeae', 'Feuil1', 'Feuil2', 'Coronatae', 'UMR_MARBEC']

Suivi_2025.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Feuil1', 'Semaestomeae', 'Coronatae', 'Staurozoa', 'UMR_MARBEC']

Suivi_2026.xlsx
['Aurelia', 'Hydrozoa', 'Cubozoa', 'Rhizostomae', 'Feuil1', 'Semaestomeae', 'Coronatae']


In [235]:
def extract_year_from_filename(path: Path) -> int | None:
    # Extrait l'année depuis un nom de fichier du type Suivi_2025.xlsx.
    match = re.search(r"(\d{4})", path.name)
    return int(match.group(1)) if match else None


def clean_text(value):
    # Nettoie une cellule texte et retourne None si elle est vide.
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def normalize_measure_type(value):
    # Normalise le type de mesure biologique.
    value = clean_text(value)
    if value is None:
        return None

    value_lower = value.lower()

    if "polype" in value_lower:
        return "polypes"
    if "éphyr" in value_lower or "ephyr" in value_lower:
        return "ephyrules"

    return value_lower


def parse_excel_file(file_path: Path) -> pd.DataFrame:
    # Transforme un fichier Excel annuel en format long.
    year = extract_year_from_filename(file_path)
    xls = pd.ExcelFile(file_path)

    rows = []

    for sheet_name in xls.sheet_names:
        # Les feuilles Feuil1 / Feuil2 sont génériques ou vides.
        if sheet_name.lower().startswith("feuil"):
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        # Les semaines sont sur la ligne 1, à partir de la colonne 4.
        week_columns = []
        for col in range(4, df.shape[1]):
            week_value = df.iloc[1, col]

            if pd.notna(week_value):
                try:
                    week_number = int(float(week_value))
                    week_columns.append((col, week_number))
                except ValueError:
                    pass

        current_species = None
        current_box = None
        current_temperature = None

        # Les données commencent à la ligne 2.
        for row_idx in range(2, df.shape[0]):
            species = clean_text(df.iloc[row_idx, 0])
            box = clean_text(df.iloc[row_idx, 1])
            temperature = df.iloc[row_idx, 2]
            measure_type = normalize_measure_type(df.iloc[row_idx, 3])

            # Si une nouvelle ligne de boîte/espèce commence, on met à jour le contexte.
            # Si la température est vide au début d'un nouveau bloc, elle reste manquante.
            new_block = species is not None or box is not None

            if species is not None:
                current_species = species

            if box is not None:
                current_box = box

            if new_block:
                current_temperature = temperature if pd.notna(temperature) else None
            elif pd.notna(temperature):
                current_temperature = temperature

            if measure_type is None:
                continue

            for col, week_number in week_columns:
                rows.append({
                    "annee": year,
                    "groupe": sheet_name,
                    "espece": current_species,
                    "code_boite": current_box,
                    "temperature_cible": current_temperature,
                    "semaine": week_number,
                    "type_mesure": measure_type,
                    "valeur": df.iloc[row_idx, col],
                    "fichier_source": file_path.name,
                    "feuille_source": sheet_name,
                    "ligne_source": row_idx + 1,
                })

    return pd.DataFrame(rows)


In [236]:
tracking_long_df = pd.concat(
    [parse_excel_file(file) for file in excel_files],
    ignore_index=True
)

display(tracking_long_df.head(20))
print("Dimensions :", tracking_long_df.shape)
print(tracking_long_df["type_mesure"].value_counts(dropna=False))


,annee,groupe,espece,code_boite,temperature_cible,semaine,type_mesure,valeur,fichier_source,feuille_source,ligne_source
0,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,1,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
1,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,2,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
2,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,3,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
3,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,4,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
4,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,5,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
5,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,6,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
6,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,7,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
7,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,8,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
8,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,9,polypes,NaN,Suivi_2019.xlsx,Aurelia,3
9,2019,Aurelia,Aurelia labiata,ALA-JKA-1.03,5,10,polypes,NaN,Suivi_2019.xlsx,Aurelia,3


Dimensions : (259688, 11)
type_mesure
polypes      130052
ephyrules    129324
nb              312
Name: count, dtype: int64


In [237]:
# Export de contrôle des lignes Staurozoa avant correction.
stauro_values_df = tracking_long_df[
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["valeur"].notna())
].copy()

stauro_values_df.to_csv(
    ANOMALY_DIR / "staurozoa_a_verifier.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes Staurozoa à vérifier :", stauro_values_df.shape)
display(stauro_values_df.head(20))


Lignes Staurozoa à vérifier : (30, 11)


,annee,groupe,espece,code_boite,temperature_cible,semaine,type_mesure,valeur,fichier_source,feuille_source,ligne_source
222995,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,20,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
222996,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,21,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
222997,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,22,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
222998,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,23,nb,3,Suivi_2025.xlsx,Staurozoa,3
222999,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,24,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
223000,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,25,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
223001,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,26,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
223002,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,27,nb,3,Suivi_2025.xlsx,Staurozoa,3
223003,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,28,nb,3.0,Suivi_2025.xlsx,Staurozoa,3
223004,2025,Staurozoa,Lucernaria quadricornis,LQU-NBE-1.01,10,29,nb,3.0,Suivi_2025.xlsx,Staurozoa,3


In [238]:
# Correction spécifique pour Staurozoa.

tracking_long_df = tracking_long_df.sort_values(
    ["fichier_source", "feuille_source", "ligne_source", "semaine"]
).copy()

stauro_mask = (
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["type_mesure"] == "nb")
)

stauro_lignes = (
    tracking_long_df[stauro_mask][["fichier_source", "feuille_source", "ligne_source"]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "ligne_source"])
    .reset_index(drop=True)
)

stauro_lignes["type_corrige"] = np.where(
    stauro_lignes.index % 2 == 0,
    "polypes",
    "ephyrules"
)

tracking_long_df = tracking_long_df.merge(
    stauro_lignes,
    on=["fichier_source", "feuille_source", "ligne_source"],
    how="left"
)

tracking_long_df["type_mesure"] = tracking_long_df["type_corrige"].fillna(
    tracking_long_df["type_mesure"]
)

tracking_long_df = tracking_long_df.drop(columns=["type_corrige"])

print(tracking_long_df["type_mesure"].value_counts(dropna=False))


type_mesure
polypes      130208
ephyrules    129480
Name: count, dtype: int64


In [239]:
clean_tracking_df = tracking_long_df.copy()

# Garder uniquement les lignes avec une valeur renseignée.
clean_tracking_df = clean_tracking_df[clean_tracking_df["valeur"].notna()].copy()

# Garder uniquement les mesures biologiques exploitables.
clean_tracking_df = clean_tracking_df[
    clean_tracking_df["type_mesure"].isin(["polypes", "ephyrules"])
].copy()

# Nettoyer les textes.
for col in ["groupe", "espece", "code_boite", "fichier_source", "feuille_source"]:
    clean_tracking_df[col] = clean_tracking_df[col].astype("string").str.strip()

clean_tracking_df["code_boite"] = (
    clean_tracking_df["code_boite"]
    .str.replace("\n", "", regex=False)
    .str.replace("\r", "", regex=False)
    .str.strip()
)

clean_tracking_df["valeur"] = pd.to_numeric(clean_tracking_df["valeur"], errors="coerce")
clean_tracking_df["temperature_cible"] = pd.to_numeric(
    clean_tracking_df["temperature_cible"],
    errors="coerce"
)

clean_tracking_df = clean_tracking_df[clean_tracking_df["valeur"].notna()].copy()

print("Données propres :", clean_tracking_df.shape)
print(clean_tracking_df["type_mesure"].value_counts(dropna=False))
print("Espèces manquantes :", clean_tracking_df["espece"].isna().sum())
print("Boîtes manquantes :", clean_tracking_df["code_boite"].isna().sum())
print("Températures manquantes :", clean_tracking_df["temperature_cible"].isna().sum())


Données propres : (76238, 11)
type_mesure
polypes      38223
ephyrules    38015
Name: count, dtype: int64
Espèces manquantes : 0
Boîtes manquantes : 0
Températures manquantes : 292


In [240]:
def extract_code_espece_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[0] if len(parts) >= 1 else None


def extract_provenance(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[1] if len(parts) >= 2 else None


def extract_numero_souche_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    if len(parts) >= 3:
        return parts[2].split(".")[0]
    return None


def extract_numero_boite_local(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[-1] if "." in code_boite else None


def extract_code_souche(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[0] if "." in code_boite else code_boite


clean_tracking_df["code_espece_local"] = clean_tracking_df["code_boite"].apply(extract_code_espece_local)
clean_tracking_df["code_souche"] = clean_tracking_df["code_boite"].apply(extract_code_souche)
clean_tracking_df["numero_souche_local"] = clean_tracking_df["code_boite"].apply(extract_numero_souche_local)
clean_tracking_df["code_provenance"] = clean_tracking_df["code_boite"].apply(extract_provenance)
clean_tracking_df["numero_boite_local"] = clean_tracking_df["code_boite"].apply(extract_numero_boite_local)

display(clean_tracking_df[[
    "code_boite",
    "code_espece_local",
    "code_souche",
    "numero_souche_local",
    "code_provenance",
    "numero_boite_local",
    "espece"
]].drop_duplicates().head(30))


,code_boite,code_espece_local,code_souche,numero_souche_local,code_provenance,numero_boite_local,espece
21528,ALA-JKA-1.02,ALA,ALA-JKA-1,1,JKA,02,Aurelia labiata
21763,ALA-JKA-1.03,ALA,ALA-JKA-1,1,JKA,03,Aurelia labiata
21988,ALA-JKA-1.04,ALA,ALA-JKA-1,1,JKA,04,Aurelia labiata
22256,ALI-JKA-1.03,ALI,ALI-JKA-1,1,JKA,03,Aurelia limbata
22408,ALI-JKA-1.04,ALI,ALI-JKA-1,1,JKA,04,Aurelia limbata
22590,ALI-JKA-2.02,ALI,ALI-JKA-2,2,JKA,02,Aurelia limbata
22776,ALI-FPA-3.01,ALI,ALI-FPA-3,3,FPA,01,Aurelia limbata
23088,AMA-JKA-1.01,AMA,AMA-JKA-1,1,JKA,01,Aurelia maldivensis
23192,AMA-JKA-1.03,AMA,AMA-JKA-1,1,JKA,03,Aurelia maldivensis
23409,AMA-JKA-1.04,AMA,AMA-JKA-1,1,JKA,04,Aurelia maldivensis


In [241]:
species_corrections = {
    "Alatina morandini": "Alatina morandinii",
    "Tiarropsis multicirrata": "Tiaropsis multicirrata",
    "Chrysaoa achlyos": "Chrysaora achlyos",
    "Cyanea lamarcki": "Cyanea lamarckii",
    "Clytia hemispherica": "Clytia hemisphaerica",
    "Cyanea nozakii Kishinouye": "Cyanea nozakii",
    "eudendryum sp": "Eudendrium sp.",
}

# Garder une trace des corrections effectuées.
species_corrections_df = pd.DataFrame([
    {"nom_initial": initial, "nom_corrige": corrected}
    for initial, corrected in species_corrections.items()
])

species_corrections_df.to_csv(
    ANOMALY_DIR / "corrections_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

clean_tracking_df["espece"] = clean_tracking_df["espece"].replace(species_corrections)

display(species_corrections_df)


,nom_initial,nom_corrige
0,Alatina morandini,Alatina morandinii
1,Tiarropsis multicirrata,Tiaropsis multicirrata
2,Chrysaoa achlyos,Chrysaora achlyos
3,Cyanea lamarcki,Cyanea lamarckii
4,Clytia hemispherica,Clytia hemisphaerica
5,Cyanea nozakii Kishinouye,Cyanea nozakii
6,eudendryum sp,Eudendrium sp.


In [242]:
# Anomalie 1 : températures manquantes.
anomalies_temperature_df = clean_tracking_df[
    clean_tracking_df["temperature_cible"].isna()
].copy()

anomalies_temperature_df.to_csv(
    ANOMALY_DIR / "anomalies_temperature_manquante.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes avec température manquante :", len(anomalies_temperature_df))

display(
    anomalies_temperature_df[[
        "fichier_source",
        "feuille_source",
        "code_boite",
        "espece",
        "annee",
        "semaine"
    ]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "code_boite", "semaine"])
    .head(30)
)


Lignes avec température manquante : 292


,fichier_source,feuille_source,code_boite,espece,annee,semaine
183664,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,1
183665,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,2
183666,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,3
183667,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,4
183668,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,5
183669,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,6
183670,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,7
183671,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,8
183672,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,9
183673,Suivi_2025.xlsx,Aurelia,ACA-FBA-1.01,Aurelia coerulea,2025,10


In [243]:
# Anomalie 2 : même code de souche associé à plusieurs espèces.
species_per_souche = (
    clean_tracking_df[["code_souche", "espece"]]
    .drop_duplicates()
    .groupby("code_souche")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

problem_souches = species_per_souche[
    species_per_souche["nb_especes"] > 1
]["code_souche"].tolist()

anomalies_souche_df = (
    clean_tracking_df[
        clean_tracking_df["code_souche"].isin(problem_souches)
    ][[
        "code_souche",
        "code_boite",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_souche", "espece", "code_boite"])
)

anomalies_souche_df.to_csv(
    ANOMALY_DIR / "anomalies_souche_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de souche associés à plusieurs espèces :", len(problem_souches))
display(species_per_souche[species_per_souche["nb_especes"] > 1])
display(anomalies_souche_df.head(50))


Codes de souche associés à plusieurs espèces : 11


,code_souche,nb_especes
32,CAC-JKA-2,2
27,AVA-TAI-1,2
197,THY-FGU-1,2
47,CFU-JKA-2,2
130,LRO-MAL-1,2
25,ATH-THA-1,2
147,OCE-TAI-1,2
183,SME-JKA-1,2
181,SMA-MAU-3,2
180,SMA-JKA-2,2


,code_souche,code_boite,espece,fichier_source,feuille_source
187453,ATH-THA-1,ATH-THA-1.01,Aurelia sp. THAI,Suivi_2025.xlsx,Aurelia
227552,ATH-THA-1,ATH-THA-1.01,Aurelia sp.4,Suivi_2026.xlsx,Aurelia
226512,AVA-TAI-1,AVA-TAI-1.01,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
226824,AVA-TAI-1,AVA-TAI-1.02,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
149701,AVA-TAI-1,AVA-TAI-1.01,"Aurelia sp.2 ""Valentine""",Suivi_2024.xlsx,Aurelia
186500,AVA-TAI-1,AVA-TAI-1.01,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
186689,AVA-TAI-1,AVA-TAI-1.02,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
99370,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2022.xlsx,Semaestomeae
134368,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2023.xlsx,Semaestomeae
172432,CAC-JKA-2,CAC-JKA-2.01,Chrysaora achlyos,Suivi_2024.xlsx,Semaestomeae


In [244]:
# Anomalie 3 : même code de boîte associé à plusieurs espèces.
species_per_boite = (
    clean_tracking_df[["code_boite", "espece"]]
    .drop_duplicates()
    .groupby("code_boite")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

codes_boites_ambigues = species_per_boite[
    species_per_boite["nb_especes"] > 1
]["code_boite"].tolist()

anomalies_boites_df = (
    clean_tracking_df[
        clean_tracking_df["code_boite"].isin(codes_boites_ambigues)
    ][[
        "code_boite",
        "code_souche",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_boite", "espece", "fichier_source"])
)

anomalies_boites_df.to_csv(
    ANOMALY_DIR / "anomalies_boites_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de boîte associés à plusieurs espèces :", len(codes_boites_ambigues))
display(species_per_boite[species_per_boite["nb_especes"] > 1])
display(anomalies_boites_df.head(50))


Codes de boîte associés à plusieurs espèces : 14


,code_boite,nb_especes
503,SME-JKA-1.01,2
123,CCP-TPR-1.08,2
497,SMA-JKA-2.06,2
496,SMA-JKA-2.05,2
504,SME-JKA-1.04,2
505,SME-JKA-1.05,2
72,AVA-TAI-1.02,2
71,AVA-TAI-1.01,2
68,ATH-THA-1.01,2
345,LRO-MAL-1.01,2


,code_boite,code_souche,espece,fichier_source,feuille_source
187453,ATH-THA-1.01,ATH-THA-1,Aurelia sp. THAI,Suivi_2025.xlsx,Aurelia
227552,ATH-THA-1.01,ATH-THA-1,Aurelia sp.4,Suivi_2026.xlsx,Aurelia
226512,AVA-TAI-1.01,AVA-TAI-1,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
149701,AVA-TAI-1.01,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2024.xlsx,Aurelia
186500,AVA-TAI-1.01,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
226824,AVA-TAI-1.02,AVA-TAI-1,Aurelia malayensis,Suivi_2026.xlsx,Aurelia
186689,AVA-TAI-1.02,AVA-TAI-1,"Aurelia sp.2 ""Valentine""",Suivi_2025.xlsx,Aurelia
173268,CCP-TPR-1.08,CCP-TPR-1,Chrysaora chesapeakei,Suivi_2024.xlsx,Semaestomeae
214552,CCP-TPR-1.08,CCP-TPR-1,"Chrysaora chesapeakei ""white""",Suivi_2025.xlsx,Semaestomeae
245336,LRO-MAL-1.01,LRO-MAL-1,Lobonemoides gracilis,Suivi_2026.xlsx,Rhizostomae


In [245]:
# ============================================================
# Séparation automatique des données valides et des anomalies
# ============================================================

# On part de toutes les données nettoyées.
df = clean_tracking_df.copy()

# 1. Codes de boîte associés à plusieurs espèces
boites_plusieurs_especes = (
    df[["code_boite", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_boite")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_boites_ambigues = boites_plusieurs_especes[
    boites_plusieurs_especes["nb_especes"] > 1
]["code_boite"].tolist()

# 2. Codes de souche associés à plusieurs espèces
souches_plusieurs_especes = (
    df[["code_souche", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_souche")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_souches_ambigues = souches_plusieurs_especes[
    souches_plusieurs_especes["nb_especes"] > 1
]["code_souche"].tolist()

# 3. Codes espèce locaux associés à plusieurs noms scientifiques
codes_especes_plusieurs_noms = (
    df[["code_espece_local", "espece"]]
    .dropna()
    .drop_duplicates()
    .groupby("code_espece_local")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
)

codes_especes_ambigus = codes_especes_plusieurs_noms[
    codes_especes_plusieurs_noms["nb_especes"] > 1
]["code_espece_local"].tolist()

# 4. Codes de boîte non standards
# Un code standard ressemble à : ABC-XXX-1.01
masque_code_boite_non_standard = ~df["code_boite"].astype(str).str.match(
    r"^[A-Z]{3}-[A-Z0-9?]{3}-[0-9]+(\.[0-9]+)?$",
    na=False
)

# 5. Températures manquantes
masque_temperature_manquante = df["temperature_cible"].isna()

# 6. Noms scientifiques associés à plusieurs codes espèce
# Exemple : Obelia sp. avec les codes OBE et OSP.
noms_plusieurs_codes = (
    df[["espece", "code_espece_local"]]
    .dropna()
    .drop_duplicates()
    .groupby("espece")["code_espece_local"]
    .nunique()
    .reset_index(name="nb_codes")
)

noms_especes_ambigus = noms_plusieurs_codes[
    noms_plusieurs_codes["nb_codes"] > 1
]["espece"].tolist()

# Masque global des anomalies
masque_anomalie = (
    df["code_boite"].isin(codes_boites_ambigues)
    | df["code_souche"].isin(codes_souches_ambigues)
    | df["code_espece_local"].isin(codes_especes_ambigus)
    | df["espece"].isin(noms_especes_ambigus)
    | masque_code_boite_non_standard
    | masque_temperature_manquante
)

# Données valides et anomalies
clean_tracking_valid_df = df[~masque_anomalie].copy()
clean_tracking_anomalies_df = df[masque_anomalie].copy()

print("Lignes totales :", len(df))
print("Lignes valides :", len(clean_tracking_valid_df))
print("Lignes anomalies :", len(clean_tracking_anomalies_df))

print("Boîtes ambiguës :", len(codes_boites_ambigues))
print("Souches ambiguës :", len(codes_souches_ambigues))
print("Codes espèce ambigus :", len(codes_especes_ambigus))
print("Codes boîte non standards :", int(masque_code_boite_non_standard.sum()))
print("Températures manquantes :", int(masque_temperature_manquante.sum()))
print("Noms scientifiques ambigus :", len(noms_especes_ambigus))

display(noms_plusieurs_codes[noms_plusieurs_codes["nb_codes"] > 1])


Lignes totales : 76238
Lignes valides : 56555
Lignes anomalies : 19683
Boîtes ambiguës : 14
Souches ambiguës : 11
Codes espèce ambigus : 16
Codes boîte non standards : 1618
Températures manquantes : 292
Noms scientifiques ambigus : 10


,espece,nb_codes
21,Aurelia sp.4,2
39,"Chrysaora chesapeakei ""white""",2
57,Cotylorhiza tuberculata,2
79,Hydrozoa sp.,4
85,Linuche aquila,3
95,Melicertum octocostatum,2
100,Obelia sp.,2
104,Pennaria disticha,2
110,Rhizostoma luteum,2
124,Stomolophus sp.2,2


In [246]:
# ============================================================
# Export des anomalies détectées
# ============================================================

ANOMALY_DIR = OUTPUT_DIR / "anomalies"
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

# Anomalies : boîtes associées à plusieurs espèces
anomalies_boites_df = df[
    df["code_boite"].isin(codes_boites_ambigues)
].copy()

anomalies_boites_df.to_csv(
    ANOMALY_DIR / "anomalies_boites_ambigues.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies : souches associées à plusieurs espèces
anomalies_souches_df = df[
    df["code_souche"].isin(codes_souches_ambigues)
].copy()

anomalies_souches_df.to_csv(
    ANOMALY_DIR / "anomalies_souches_ambigues.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies : codes espèce associés à plusieurs noms
anomalies_especes_df = df[
    df["code_espece_local"].isin(codes_especes_ambigus)
].copy()

anomalies_especes_df.to_csv(
    ANOMALY_DIR / "anomalies_codes_especes_ambigus.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies : noms scientifiques associés à plusieurs codes espèce
anomalies_noms_especes_df = df[
    df["espece"].isin(noms_especes_ambigus)
].copy()

anomalies_noms_especes_df.to_csv(
    ANOMALY_DIR / "anomalies_noms_especes_ambigus.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies : codes boîte non standards
anomalies_codes_non_standards_df = df[
    masque_code_boite_non_standard
].copy()

anomalies_codes_non_standards_df.to_csv(
    ANOMALY_DIR / "anomalies_codes_boite_non_standards.csv",
    index=False,
    encoding="utf-8-sig"
)

# Anomalies : températures manquantes
anomalies_temperatures_df = df[
    masque_temperature_manquante
].copy()

anomalies_temperatures_df.to_csv(
    ANOMALY_DIR / "anomalies_temperatures_manquantes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Fichier global de toutes les anomalies
clean_tracking_anomalies_df.to_csv(
    ANOMALY_DIR / "anomalies_toutes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fichiers anomalies exportés dans :", ANOMALY_DIR)
print("Toutes les anomalies :", len(clean_tracking_anomalies_df))


Fichiers anomalies exportés dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies
Toutes les anomalies : 19683


In [247]:
# Table ESPECE, à partir des données valides.

espece_df = (
    clean_tracking_valid_df[[
        "espece",
        "code_espece_local"
    ]]
    .dropna(subset=["espece"])
    .drop_duplicates()
    .sort_values("espece")
    .reset_index(drop=True)
)

espece_df["id_espece"] = range(1, len(espece_df) + 1)

espece_df = espece_df.rename(columns={
    "espece": "nom_scientifique",
    "code_espece_local": "code_espece"
})

espece_df = espece_df[[
    "id_espece",
    "code_espece",
    "nom_scientifique"
]]

print("Nombre d'espèces :", len(espece_df))
print("Doublons nom scientifique :", espece_df["nom_scientifique"].duplicated().sum())
print("Doublons code espèce :", espece_df["code_espece"].duplicated().sum())
display(espece_df.head(20))


Nombre d'espèces : 101
Doublons nom scientifique : 0
Doublons code espèce : 0


,id_espece,code_espece,nom_scientifique
0,1,AFL,Acromitus flagellatus
1,2,AHA,Acromitus hardenbergi
2,3,ACO,Aequorea coerulescens
3,4,AMD,Aequorea macrodactyla
4,5,AVI,Aequorea victoria
5,6,ADI,Aglaophenia diegensis
6,7,AAL,Alatina alata
7,8,AMO,Alatina morandinii
8,9,ASH,Anomalorhiza shawi
9,10,AAU,Aurelia aurita


In [248]:
souche_df = (
    clean_tracking_valid_df[[
        "code_souche",
        "numero_souche_local",
        "code_provenance",
        "espece"
    ]]
    .dropna(subset=["code_souche", "espece"])
    .drop_duplicates()
    .reset_index(drop=True)
)

souche_df = souche_df.merge(
    espece_df[["id_espece", "nom_scientifique"]],
    left_on="espece",
    right_on="nom_scientifique",
    how="left"
)

souche_df["id_souche"] = range(1, len(souche_df) + 1)

souche_df = souche_df[[
    "id_souche",
    "code_souche",
    "numero_souche_local",
    "code_provenance",
    "id_espece"
]]

display(souche_df.head(20))
print("Nombre de souches :", len(souche_df))
print("Souches sans espèce :", souche_df["id_espece"].isna().sum())


,id_souche,code_souche,numero_souche_local,code_provenance,id_espece
0,1,ALA-JKA-1,1,JKA,13
1,2,ALI-JKA-1,1,JKA,14
2,3,ALI-JKA-2,2,JKA,14
3,4,ALI-FPA-3,3,FPA,14
4,5,AMA-JKA-1,1,JKA,15
5,6,LUN-ESS-1,1,ESS,66
6,7,LUN-ABR-2,2,ABR,66
7,8,AMO-JKA-1,1,JKA,8
8,9,CMA-JKA-1,1,JKA,22
9,10,ACO-JKA-2,2,JKA,3


Nombre de souches : 140
Souches sans espèce : 0


In [249]:
boite_df = (
    clean_tracking_valid_df[[
        "code_boite",
        "code_souche",
        "numero_boite_local",
        "espece"
    ]]
    .dropna(subset=["code_boite", "code_souche", "espece"])
    .drop_duplicates()
    .reset_index(drop=True)
)

boite_df = boite_df.merge(
    espece_df[["id_espece", "nom_scientifique"]],
    left_on="espece",
    right_on="nom_scientifique",
    how="left"
)

boite_df = boite_df.merge(
    souche_df[["id_souche", "code_souche", "id_espece"]],
    on=["code_souche", "id_espece"],
    how="left"
)

boite_df["id_boite"] = range(1, len(boite_df) + 1)
boite_df["code_local"] = boite_df["code_boite"]

boite_df = boite_df[[
    "id_boite",
    "code_local",
    "numero_boite_local",
    "id_souche"
]]

display(boite_df.head(20))
print("Nombre de boîtes :", len(boite_df))
print("Doublons code_local :", boite_df["code_local"].duplicated().sum())
print("Boîtes sans souche :", boite_df["id_souche"].isna().sum())


,id_boite,code_local,numero_boite_local,id_souche
0,1,ALA-JKA-1.02,02,1
1,2,ALA-JKA-1.03,03,1
2,3,ALA-JKA-1.04,04,1
3,4,ALI-JKA-1.03,03,2
4,5,ALI-JKA-1.04,04,2
5,6,ALI-JKA-2.02,02,3
6,7,ALI-FPA-3.01,01,4
7,8,AMA-JKA-1.01,01,5
8,9,AMA-JKA-1.03,03,5
9,10,AMA-JKA-1.04,04,5


Nombre de boîtes : 397
Doublons code_local : 0
Boîtes sans souche : 0


In [250]:
# Table SAISIR_RELEVE, à partir des données valides uniquement.

releve_base_df = clean_tracking_valid_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

print("Lignes de relevés sans boîte :", releve_base_df["id_boite"].isna().sum())

releve_df = (
    releve_base_df
    .pivot_table(
        index=["annee", "semaine", "id_boite"],
        columns="type_mesure",
        values="valeur",
        aggfunc="first"
    )
    .reset_index()
)

releve_df = releve_df.rename(columns={
    "polypes": "nombre_polypes",
    "ephyrules": "nombre_ephyrules"
})

releve_df["date_releve"] = (
    releve_df["annee"].astype(int).astype(str)
    + "-S"
    + releve_df["semaine"].astype(int).astype(str).str.zfill(2)
)

releve_df = releve_df[[
    "id_boite",
    "annee",
    "semaine",
    "date_releve",
    "nombre_polypes",
    "nombre_ephyrules"
]]

display(releve_df.head(20))
print("Nombre de relevés :", len(releve_df))
print("Relevés sans boîte :", releve_df["id_boite"].isna().sum())
print("Relevés sans polypes :", releve_df["nombre_polypes"].isna().sum())
print("Relevés sans éphyrules :", releve_df["nombre_ephyrules"].isna().sum())


Lignes de relevés sans boîte : 0


type_mesure,id_boite,annee,semaine,date_releve,nombre_polypes,nombre_ephyrules
0,1,2020,1,2020-S01,200.0,200.0
1,2,2020,1,2020-S01,50.0,0.0
2,4,2020,1,2020-S01,200.0,0.0
3,6,2020,1,2020-S01,100.0,0.0
4,7,2020,1,2020-S01,200.0,200.0
5,8,2020,1,2020-S01,200.0,0.0
6,9,2020,1,2020-S01,200.0,0.0
7,11,2020,1,2020-S01,200.0,0.0
8,12,2020,1,2020-S01,2.0,0.0
9,13,2020,1,2020-S01,200.0,3.0


Nombre de relevés : 28270
Relevés sans boîte : 0
Relevés sans polypes : 69
Relevés sans éphyrules : 192


In [251]:
# Anomalie 4 : relevés incomplets.
anomalies_releves_incomplets_df = releve_df[
    releve_df["nombre_polypes"].isna() | releve_df["nombre_ephyrules"].isna()
].copy()

anomalies_releves_incomplets_df.to_csv(
    ANOMALY_DIR / "anomalies_releves_incomplets.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Relevés incomplets :", len(anomalies_releves_incomplets_df))
display(anomalies_releves_incomplets_df.head(20))


Relevés incomplets : 261


type_mesure,id_boite,annee,semaine,date_releve,nombre_polypes,nombre_ephyrules
6372,90,2021,41,2021-S41,NaN,0.0
6448,90,2021,42,2021-S42,NaN,0.0
6524,90,2021,43,2021-S43,NaN,0.0
6582,67,2021,44,2021-S44,NaN,0.0
6599,90,2021,44,2021-S44,NaN,0.0
6658,67,2021,45,2021-S45,NaN,0.0
6675,90,2021,45,2021-S45,NaN,0.0
6734,67,2021,46,2021-S46,NaN,0.0
6751,90,2021,46,2021-S46,NaN,0.0
6810,67,2021,47,2021-S47,NaN,0.0


In [252]:
# Table ZONE_THERMIQUE, à partir des données valides uniquement.

zone_thermique_df = (
    clean_tracking_valid_df[["temperature_cible"]]
    .dropna()
    .drop_duplicates()
    .sort_values("temperature_cible")
    .reset_index(drop=True)
)

zone_thermique_df["id_zone"] = range(1, len(zone_thermique_df) + 1)
zone_thermique_df["nom_zone"] = (
    "Zone "
    + zone_thermique_df["temperature_cible"].astype(str)
    + "°C"
)

zone_thermique_df = zone_thermique_df[[
    "id_zone",
    "nom_zone",
    "temperature_cible"
]]

display(zone_thermique_df)
print("Nombre de zones thermiques :", len(zone_thermique_df))


,id_zone,nom_zone,temperature_cible
0,1,Zone 5.0°C,5.0
1,2,Zone 10.0°C,10.0
2,3,Zone 15.0°C,15.0
3,4,Zone 20.0°C,20.0
4,5,Zone 25.0°C,25.0
5,6,Zone 30.0°C,30.0


Nombre de zones thermiques : 6


In [253]:
# Export des lignes valides biologiquement mais sans température cible.
anomalies_temperature_valid_df = clean_tracking_valid_df[
    clean_tracking_valid_df["temperature_cible"].isna()
].copy()

anomalies_temperature_valid_df.to_csv(
    ANOMALY_DIR / "anomalies_temperature_manquante_valides.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes valides avec température manquante :", len(anomalies_temperature_valid_df))
display(anomalies_temperature_valid_df[[
    "code_boite",
    "espece",
    "annee",
    "semaine",
    "fichier_source",
    "feuille_source"
]].drop_duplicates().head(30))


Lignes valides avec température manquante : 0


,code_boite,espece,annee,semaine,fichier_source,feuille_source


In [254]:
# Table RANGE, à partir des données valides avec température renseignée.

range_base_df = clean_tracking_valid_df[
    clean_tracking_valid_df["temperature_cible"].notna()
].copy()

range_base_df = range_base_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

range_base_df = range_base_df.merge(
    zone_thermique_df[["id_zone", "temperature_cible"]],
    on="temperature_cible",
    how="left"
)

range_df = (
    range_base_df[[
        "id_boite",
        "id_zone",
        "annee",
        "semaine"
    ]]
    .dropna(subset=["id_boite", "id_zone"])
    .drop_duplicates()
    .reset_index(drop=True)
)

range_df["date_entree"] = (
    range_df["annee"].astype(int).astype(str)
    + "-S"
    + range_df["semaine"].astype(int).astype(str).str.zfill(2)
)

range_df = range_df[[
    "id_boite",
    "id_zone",
    "annee",
    "semaine",
    "date_entree"
]]

display(range_df.head(20))
print("Nombre de rangements :", len(range_df))
print("Rangements sans boîte :", range_df["id_boite"].isna().sum())
print("Rangements sans zone :", range_df["id_zone"].isna().sum())


,id_boite,id_zone,annee,semaine,date_entree
0,1,1,2020,1,2020-S01
1,1,1,2020,2,2020-S02
2,1,1,2020,3,2020-S03
3,1,1,2020,4,2020-S04
4,1,1,2020,5,2020-S05
5,1,1,2020,6,2020-S06
6,1,1,2020,7,2020-S07
7,1,1,2020,8,2020-S08
8,1,1,2020,9,2020-S09
9,1,1,2020,10,2020-S10


Nombre de rangements : 28299
Rangements sans boîte : 0
Rangements sans zone : 0


In [255]:
espece_df.to_csv(OUTPUT_DIR / "espece.csv", index=False, encoding="utf-8-sig")
souche_df.to_csv(OUTPUT_DIR / "souche.csv", index=False, encoding="utf-8-sig")
boite_df.to_csv(OUTPUT_DIR / "boite.csv", index=False, encoding="utf-8-sig")
releve_df.to_csv(OUTPUT_DIR / "saisir_releve.csv", index=False, encoding="utf-8-sig")
zone_thermique_df.to_csv(OUTPUT_DIR / "zone_thermique.csv", index=False, encoding="utf-8-sig")
range_df.to_csv(OUTPUT_DIR / "range.csv", index=False, encoding="utf-8-sig")

print("Exports terminés dans :", OUTPUT_DIR)
print("Anomalies exportées dans :", ANOMALY_DIR)


Exports terminés dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase
Anomalies exportées dans : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies


In [256]:
# ============================================================
# Vérification finale avant import Django
# ============================================================

resume_controles = {
    "nb_especes": len(espece_df),
    "nb_souches": len(souche_df),
    "nb_boites_valides": len(boite_df),
    "nb_releves": len(releve_df),
    "nb_zones_thermiques": len(zone_thermique_df),
    "nb_rangements": len(range_df),
    "doublons_code_espece": int(espece_df["code_espece"].duplicated().sum()),
    "doublons_nom_scientifique": int(espece_df["nom_scientifique"].duplicated().sum()),
    "doublons_code_souche": int(souche_df["code_souche"].duplicated().sum()),
    "doublons_code_local": int(boite_df["code_local"].duplicated().sum()),
    "souches_sans_espece": int(souche_df["id_espece"].isna().sum()),
    "boites_sans_souche": int(boite_df["id_souche"].isna().sum()),
    "releves_sans_boite": int(releve_base_df["id_boite"].isna().sum()),
    "rangements_sans_boite": int(range_df["id_boite"].isna().sum()),
    "rangements_sans_zone": int(range_df["id_zone"].isna().sum()),
    "anomalies_total": len(clean_tracking_anomalies_df),
    "anomalies_temperature": len(anomalies_temperatures_df),
    "anomalies_boites_plusieurs_especes": len(codes_boites_ambigues),
    "anomalies_souches_plusieurs_especes": len(codes_souches_ambigues),
    "anomalies_codes_especes_ambigus": len(codes_especes_ambigus),
    "anomalies_noms_especes_ambigus": len(noms_especes_ambigus),
    "anomalies_releves_incomplets": len(anomalies_releves_incomplets_df),
}

resume_controles_df = pd.DataFrame(
    resume_controles.items(),
    columns=["controle", "valeur"]
)

display(resume_controles_df)


,controle,valeur
0,nb_especes,101
1,nb_souches,140
2,nb_boites_valides,397
3,nb_releves,28270
4,nb_zones_thermiques,6
5,nb_rangements,28299
6,doublons_code_espece,0
7,doublons_nom_scientifique,0
8,doublons_code_souche,0
9,doublons_code_local,0


In [257]:
# Les CSV finaux sont dans OUTPUT_DIR.
# Les anomalies exclues sont dans ANOMALY_DIR.
print("CSV finaux :", OUTPUT_DIR)
print("Fichiers anomalies :", ANOMALY_DIR)


CSV finaux : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase
Fichiers anomalies : /Users/akkouh/Desktop/POLYPBASE/exports_polypbase/anomalies


In [258]:
# Vérification rapide des fichiers exportés.
for csv_file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(csv_file.name)

print("--- anomalies ---")
for csv_file in sorted(ANOMALY_DIR.glob("*.csv")):
    print(csv_file.name)


boite.csv
espece.csv
range.csv
saisir_releve.csv
souche.csv
zone_thermique.csv
--- anomalies ---
anomalies_boites_ambigues.csv
anomalies_boites_plusieurs_especes.csv
anomalies_codes_boite_non_standards.csv
anomalies_codes_especes_ambigus.csv
anomalies_noms_especes_ambigus.csv
anomalies_releves_incomplets.csv
anomalies_souche_plusieurs_especes.csv
anomalies_souches_ambigues.csv
anomalies_temperature_manquante.csv
anomalies_temperature_manquante_valides.csv
anomalies_temperatures_manquantes.csv
anomalies_toutes.csv
corrections_noms_especes.csv
staurozoa_a_verifier.csv
